# Context Enrichment Window

## Overview

Standard retrieval returns isolated chunks — they may start or end mid-sentence, missing important context. The **context enrichment window** technique pads each retrieved chunk with its **neighboring chunks** from the original document, giving the LLM a wider view.

| Standard Retrieval | Context Enrichment |
|---|---|
| Returns the matched chunk only | Returns the matched chunk **+ neighbors before/after** |
| May cut off mid-sentence | Provides a complete, coherent window |
| Fixed-size context | Wider, more informative context |

## How It Works

1. Split document into chunks **with overlap**, tagging each with its sequential index
2. Retrieve the most relevant chunk via vector search
3. Look up the chunk's index, then fetch the chunks before and after it
4. Concatenate them (accounting for overlap) into one expanded context window

## Models Used

- **Embeddings**: `mxbai-embed-large:335m` via Ollama

<div style="text-align: center;">
<img src="./images/vector-search-comparison_context_enrichment.svg" alt="Context Enrichment" style="width:70%; height:auto;">
</div>

<div style="text-align: center;">
<img src="./images/context_enrichment_window.svg" alt="Context Enrichment Window" style="width:70%; height:auto;">
</div>

---
## Step 0: Import Packages

In [ ]:
import fitz
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings

---
## Step 1: Set Up the Embedding Model

In [ ]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("Embedding model ready")

---
## Step 2: Load the PDF as Text

In [ ]:
path = "data/Understanding_Climate_Change.pdf"

doc = fitz.open(path)
content = ""
for page_num in range(len(doc)):
    content += doc[page_num].get_text()

print(f"Loaded {len(doc)} pages, {len(content)} characters")

---
## Step 3: Split into Chunks with Index Metadata

Each chunk is tagged with its sequential `index` so we can look up neighboring chunks later. We use **overlapping chunks** — the overlap will be accounted for when concatenating neighbors.

In [ ]:
chunk_size = 400
chunk_overlap = 200

chunks = []
start = 0
idx = 0
while start < len(content):
    end = start + chunk_size
    chunk_text = content[start:end]
    chunks.append(Document(page_content=chunk_text, metadata={"index": idx}))
    idx += 1
    start += chunk_size - chunk_overlap

print(f"Created {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")
print(f"Each chunk has metadata: {chunks[0].metadata}")

---
## Step 4: Build the Vector Store and Retriever

In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# Build a lookup dict: index -> chunk text (for fast neighbor retrieval)
chunk_lookup = {chunk.metadata["index"]: chunk.page_content for chunk in chunks}

print(f"Vector store created with {len(chunks)} chunks")
print(f"Chunk lookup table built ({len(chunk_lookup)} entries)")

---
## Step 5: Compare Baseline vs. Context-Enriched Retrieval

For each retrieved chunk, we look up its index and fetch `num_neighbors` chunks before and after. The chunks are concatenated, trimming the overlap regions so the text reads naturally.

In [ ]:
query = "Explain the role of deforestation and fossil fuels in climate change."
num_neighbors = 1  # 1 chunk before + 1 chunk after

print(f"Query: {query}\n")

# --- Baseline: just the retrieved chunk ---
baseline_results = retriever.invoke(query)
baseline_chunk = baseline_results[0]
print("=== Baseline Retrieval ===")
print(baseline_chunk.page_content)

# --- Context-enriched: add neighboring chunks ---
current_index = baseline_chunk.metadata["index"]
start_index = max(0, current_index - num_neighbors)
end_index = min(len(chunk_lookup) - 1, current_index + num_neighbors)

# Gather neighbor chunks in order
neighbor_texts = []
for i in range(start_index, end_index + 1):
    if i in chunk_lookup:
        neighbor_texts.append(chunk_lookup[i])

# Concatenate, trimming overlap between consecutive chunks
enriched_text = neighbor_texts[0]
for i in range(1, len(neighbor_texts)):
    # Remove the overlapping part from the end of the accumulated text
    overlap_start = max(0, len(enriched_text) - chunk_overlap)
    enriched_text = enriched_text[:overlap_start] + neighbor_texts[i]

print(f"\n=== Context-Enriched Retrieval (chunk {start_index} to {end_index}) ===")
print(enriched_text)

print(f"\nBaseline length: {len(baseline_chunk.page_content)} chars")
print(f"Enriched length: {len(enriched_text)} chars")

---
## Step 6: Another Example — AI History Document

Let's try a second example with a shorter document about AI history. The query is about when deep learning became prominent — the answer spans across chunk boundaries.

In [ ]:
ai_content = """Artificial Intelligence (AI) has a rich history dating back to the mid-20th century. The term "Artificial Intelligence" was coined in 1956 at the Dartmouth Conference, marking the field's official beginning.

In the 1950s and 1960s, AI research focused on symbolic methods and problem-solving. The Logic Theorist, created in 1955 by Allen Newell and Herbert A. Simon, is often considered the first AI program.

The 1960s saw the development of expert systems, which used predefined rules to solve complex problems. DENDRAL, created in 1965, was one of the first expert systems, designed to analyze chemical compounds.

However, the 1970s brought the first "AI Winter," a period of reduced funding and interest in AI research, largely due to overpromised capabilities and underdelivered results.

The 1980s saw a resurgence with the popularization of expert systems in corporations. The Japanese government's Fifth Generation Computer Project also spurred increased investment in AI research globally.

Neural networks gained prominence in the 1980s and 1990s. The backpropagation algorithm, although discovered earlier, became widely used for training multi-layer networks during this time.

The late 1990s and 2000s marked the rise of machine learning approaches. Support Vector Machines (SVMs) and Random Forests became popular for various classification and regression tasks.

Deep Learning, a subset of machine learning using neural networks with many layers, began to show promising results in the early 2010s. The breakthrough came in 2012 when a deep neural network significantly outperformed other machine learning methods in the ImageNet competition.

Since then, deep learning has revolutionized many AI applications, including image and speech recognition, natural language processing, and game playing. In 2016, Google's AlphaGo defeated a world champion Go player, a landmark achievement in AI.

The current era of AI is characterized by the integration of deep learning with other AI techniques, the development of more efficient and powerful hardware, and the ethical considerations surrounding AI deployment.

Transformers, introduced in 2017, have become a dominant architecture in natural language processing, enabling models like GPT (Generative Pre-trained Transformer) to generate human-like text.

As AI continues to evolve, new challenges and opportunities arise. Explainable AI, robust and fair machine learning, and artificial general intelligence (AGI) are among the key areas of current and future research in the field.
"""

# Split with smaller chunks to better demonstrate the effect
ai_chunk_size = 250
ai_overlap = 20

ai_chunks = []
start = 0
idx = 0
while start < len(ai_content):
    end = start + ai_chunk_size
    ai_chunks.append(Document(page_content=ai_content[start:end], metadata={"index": idx}))
    idx += 1
    start += ai_chunk_size - ai_overlap

ai_vectorstore = FAISS.from_documents(ai_chunks, embedding_model)
ai_retriever = ai_vectorstore.as_retriever(search_kwargs={"k": 1})
ai_lookup = {c.metadata["index"]: c.page_content for c in ai_chunks}

print(f"AI document: {len(ai_chunks)} chunks (size={ai_chunk_size}, overlap={ai_overlap})")

In [ ]:
ai_query = "When did deep learning become prominent in AI?"
print(f"Query: {ai_query}\n")

# --- Baseline ---
ai_baseline = ai_retriever.invoke(ai_query)
print("=== Baseline Retrieval ===")
print(ai_baseline[0].page_content)

# --- Context-enriched ---
ai_idx = ai_baseline[0].metadata["index"]
ai_start = max(0, ai_idx - 1)
ai_end = min(len(ai_lookup) - 1, ai_idx + 1)

ai_neighbors = [ai_lookup[i] for i in range(ai_start, ai_end + 1) if i in ai_lookup]

ai_enriched = ai_neighbors[0]
for i in range(1, len(ai_neighbors)):
    overlap_start = max(0, len(ai_enriched) - ai_overlap)
    ai_enriched = ai_enriched[:overlap_start] + ai_neighbors[i]

print(f"\n=== Context-Enriched Retrieval (chunks {ai_start} to {ai_end}) ===")
print(ai_enriched)

print(f"\nBaseline: {len(ai_baseline[0].page_content)} chars")
print(f"Enriched: {len(ai_enriched)} chars")
print("\nThe enriched version includes the preceding context (SVMs, Random Forests)")
print("and the following context (AlphaGo, image recognition), giving a complete picture.")

---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up embedding model |
| 2 | Loaded PDF as text |
| 3 | Split into overlapping chunks, each tagged with its sequential index |
| 4 | Built FAISS vector store + chunk lookup table |
| 5 | **Compared** baseline (single chunk) vs. context-enriched (chunk + neighbors) |
| 6 | **Second example** with AI history document — showed enrichment captures cross-chunk answers |

**Key insight:** Context enrichment is simple but powerful. By tagging each chunk with its index and keeping a lookup table, you can expand any retrieved chunk into a wider window. This catches information that straddles chunk boundaries — like when the question is about "deforestation and fossil fuels" but each topic sits in a different chunk. The enriched window captures both.